# استماع أعمى: أي محرك يقص الآيات أحسن؟ · Blind listening: which engine cuts ayahs best?

لكل آية تسمع ثلاث قصّات من نفس ملف الصوت: محرك عثمان، ومنجّم (whisperx)، وتوقيت mp3quran.
تظهر بأسماء «أ، ب، ج» بترتيب عشوائي، وتنكشف بعد ما تختار. كل صوت يُحفظ فوراً في Drive (`votes.json`).
For every ayah you hear three cuts of the same audio, labelled A/B/C in random order, revealed after you vote.

- يكفي وقت التشغيل العادي بالمعالج (CPU). يقرأ ملفات النوتبوكين الآخرين من Drive، فشغّلهم قبله.
- الصوت من رابط mp3quran مباشرة، أو من ملفاتك في Drive إذا كان المصحف عندك.
- **الهامش (`PAD`):** المحركان يبدآن الآية مع أول صوت؛ هنا تمتد كل آية في السكتة قبلها وبعدها بنصفها (بحد أقصى `PAD` ثانية)
  حتى تُسمع مثل mp3quran الذي يقسم السكتة بين الآيتين. توقيت mp3quran كما هو.
- كل الآيات المختلف عليها (فرق أكثر من `THRESHOLD` ثانية) مع عينة عشوائية من الباقي، مخلوطة بترتيب عشوائي.
- إذا تساوى أكثر من مصدر في الأفضلية اخترهم كلهم: الفائزون يتقاسمون نقطة الآية.
- الأصوات تُحفظ في `votes-pad<PAD>.json`، فكل هامش له جولة مستقلة. **إذا انقطعت الجلسة:** Run all، والآيات المصوَّت عليها لا ترجع.

In [ ]:
OUT_DIR = "/content/drive/MyDrive/othman-timings"
RECITERS = ""      # فاضي = الكل، أو مثلاً "abdulrahman-sudais_r1, maher-muaiqly_r3"
SURAHS = ""        # فاضي = الكل، أو مثلاً "67,78"
THRESHOLD = 0.5    # أقل فرق بالثواني يخلي الآية «مختلف عليها»
RANDOM_SHARE = 0.2 # نسبة العينة العشوائية من الآيات المتقاربة
PAD = 0.5          # أقصى هامش بالثواني قبل الآية وبعدها لمحركنا ومنجّم (0 = بلا هامش)

In [ ]:
import os
import sys
from google.colab import drive, userdata


def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None


drive.mount("/content/drive")
BRANCH = "main"
# GH_TOKEN مطلوب فقط ما دام المستودع خاصاً · only needed while the repository is private
gh = secret("GH_TOKEN")
repo = f"https://{gh}@github.com/vvd7mvv/othman-engine" if gh else "https://github.com/vvd7mvv/othman-engine"
!rm -rf /content/othman-engine && git clone -q --depth 1 -b {BRANCH} {repo} /content/othman-engine
sys.path.insert(0, "/content/othman-engine/notebooks")
import timings

## الصفحة · The page

In [ ]:
from pathlib import Path

from google.colab import output
from IPython.display import HTML, display

votes_path = Path(OUT_DIR) / f"votes-pad{PAD}.json"
votes = timings.load_votes(votes_path)
queue = timings.listen_queue(OUT_DIR, RECITERS, SURAHS, THRESHOLD, RANDOM_SHARE, PAD)
by_id = {item["id"]: item for item in queue}
pending = [dict(item) for item in queue if item["id"] not in votes]
# ملفات Drive: خادم صغير داخل الجلسة، ورابطه العام من Colab
local_files = timings.serve_local_audio(pending)
if local_files:
    timings.use_served_audio(pending, local_files, output.eval_js("google.colab.kernel.proxyPort(8765)"))


def save_vote(item_id, winners, order):
    timings.record_vote(votes_path, votes, by_id[item_id], list(winners), list(order))


output.register_callback("othman.vote", save_vote)
disputed = sum(item["bucket"] == "disputed" for item in queue)
print(f"{len(queue)} ayahs ({disputed} disputed, {len(queue) - disputed} random), "
      f"{len(queue) - len(pending)} voted before, {len(pending)} to go")
if not queue:
    print("ما لقيت آيات. تأكد إن نوتبوك محرك عثمان ونوتبوك منجّم (whisperx) خلصوا على نفس السور.")
display(HTML(timings.listen_page(pending)))

## النتيجة · Results
شغّلها متى ما تبي (حتى قبل ما تخلص). كل آية نقطة واحدة يتقاسمها من اخترتهم، والنسبة من مجموع النقاط.

In [ ]:
import pandas as pd

votes = timings.load_votes(votes_path)
points = pd.DataFrame(timings.vote_points(votes))
if points.empty:
    raise SystemExit("ما فيه أصوات بعد · No votes yet")
names = {"othman-engine": "محرك عثمان", "munajjam-whisperx": "منجّم whisperx", "mp3quran": "mp3quran"}
points["source"] = points["source"].map(names)
all_equal = sum(len(v["winners"]) == len(v["order"]) for v in votes.values())
print(f"{len(votes)} votes ({votes_path.name}), all sources equal in {all_equal}")
total = points.groupby("source")["points"].sum()
display(pd.DataFrame({"% of points": (100 * total / total.sum()).round(1), "points": total.round(1)})
        .sort_values("points", ascending=False))
for column, title in (("reciter", "% per reciter"), ("bucket", "% disputed / random")):
    table = points.pivot_table(index=column, columns="source", values="points", aggfunc="sum", fill_value=0)
    display((100 * table.div(table.sum(axis=1), axis=0)).round(1).rename_axis(columns=title))